In [ ]:
# 1. Instalación de librerías necesarias
!pip install roboflow ultralytics -q

import torch
from roboflow import Roboflow
from ultralytics import YOLO

# 2. Configuración inicial y de dispositivo 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# 3. Adquisición y Preparación de Datos
rf = Roboflow(api_key="HjJmS7s5cb6TSDX23SuO")
project = rf.workspace("matiass-workspace-svynh").project("fire-smoke-detection-yolov11-y53yn-gmrm1")
version = project.version(1)
# Descargamos directamente en formato compatible con YOLOv8
dataset = version.download("yolov8")
# 4. Inicialización del Modelo Base
# YOLOv8 incorpora internamente el Backbone, Neck y Head
model = YOLO('yolov8l.pt')  

# 5. Lógica de Entrenamiento
# El parámetro 'data' usa directamente el archivo de configuración descargado
print("\n[INICIANDO ENTRENAMIENTO]")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=80,                  # Número de épocas a iterar
    imgsz=640,                  # Resolución de las imágenes 
    batch=32,                   # Tamaño de lote
    device=[0, 1],
    plots=True                  # Genera curvas de pérdidas y métricas
)

# 6. Evaluación Formal
print("\n[RESULTADOS FINALES EN SET DE VALIDACIÓN]")
metrics = model.val(split='test')
print("\n================ METRICAS GLOBALES ================")
print(f"mAP@50 (Precisión General Promedio): {metrics.box.map50:.4f}")
print(f"mAP@50-95 (Precisión bajo criterios estrictos): {metrics.box.map:.4f}")

# Reporte de Métricas Específicas por Clase
print("\n================ METRICAS POR CLASE ================")
for i, name in enumerate(metrics.names.values()):
    print(f"Clase [{name.upper()}]:")
    print(f"  Precision (Exactitud de las predicciones): {metrics.box.class_result(i)[0]:.4f}")
    print(f"  Recall (Capacidad de detección / Sensibilidad): {metrics.box.class_result(i)[1]:.4f}")

Usando dispositivo: cuda
loading Roboflow workspace...
loading Roboflow project...

[INICIANDO ENTRENAMIENTO]
Ultralytics 8.4.69 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                       CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Fire-Smoke-Detection-Yolov11-1/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, m